# Return-v4 parallel training — C-N, seeds [552]

Add Input **BS5_RETURN_V4_PARALLEL_DATASET.zip**, chọn GPU rồi Run All.
Mọi notebook dùng CHUNG một source Dataset; output riêng cho `CN_552`.
Notebook này chỉ train 1 run đã gán: mỗi run60k train +66k validation.
Các run khác chạy ở notebook khác; không chờ và không đọc output của nhau khi train.

Mặc định dùng4 notebook `CR_551`, `CR_552`, `CN_551`, `CN_552` song song nếu tài khoản được cấp đủ phiên GPU.
Nếu chỉ có2 phiên, dùng `CR_551_552` và `CN_551_552`; mỗi notebook train hai seed tuần tự.
Chỉ chọn MỘT cách chia, không chạy cả hai cách vì sẽ trùng job.

Source giữ nguyên env Return-v4, learner, reward, curriculum và lịch seed/UE trace.
GPU chỉ giúp một phần; tốc độ/giới hạn phiên phụ thuộc tài nguyên thực tế.
Khởi chạy sẽ báo lỗi sớm nếu GPU hoặc PyTorch CUDA chưa sẵn sàng, không âm thầm dùng CPU.

In [ ]:
from pathlib import Path
import time
SESSION_START=time.monotonic()
SOURCE_PATH=None
RESUME_INPUT_DIR=None
TRAIN_ARMS=['C-N']
TRAINING_SEEDS=[552]
JOB_LABEL='CN_552'
OUTPUT_ROOT=Path('/kaggle/working/bs5_v4_train_CN_552')
DEVICE='cuda'
REQUIRE_GPU=True  # CPU có chủ đích: đặt False và DEVICE='cpu' TRƯỚC prepare
MAX_TOTAL_SESSION_SECONDS=10.5*3600
EXPORT_RESERVE_SECONDS=30*60
RUN_MAIN=True

In [ ]:
import os, sys, json, zipfile, hashlib, shutil, subprocess, time
from pathlib import Path
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.dont_write_bytecode = True
if 'SESSION_START' not in globals(): SESSION_START = time.monotonic()
INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
MARKER = 'configs/bs5_return_v4_fresh_train.json'
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def is_main_archive(path, version):
    try:
        with zipfile.ZipFile(path) as z:
            names=[n for n in z.namelist() if len(Path(n).parts)==2 and Path(n).name=='protocol.json']
            if len(names)!=1: return False
            p=json.loads(z.read(names[0]))
            return p.get('version')==version and not p.get('diagnostic',False)
    except (OSError,ValueError,zipfile.BadZipFile): return False
def extract_safe(archive, target):
    target=Path(target).resolve()
    for m in archive.infolist():
        dest=(target/m.filename).resolve()
        if not dest.is_relative_to(target) or (m.external_attr>>16)&0o170000==0o120000:
            raise RuntimeError('Unsafe ZIP member: '+m.filename)
    if archive.testzip() is not None: raise RuntimeError('ZIP CRC failed')
    archive.extractall(target)
if SOURCE_PATH is None:
    choices=[]
    for m in INPUT.rglob('DATASET_MANIFEST.json'):
        if (m.parent/MARKER).exists(): choices.append(m.parent)
    for f in INPUT.rglob('*.zip'):
        try:
            with zipfile.ZipFile(f) as z:
                if 'uav_thesis/DATASET_MANIFEST.json' in z.namelist() and 'uav_thesis/'+MARKER in z.namelist(): choices.append(f)
        except zipfile.BadZipFile: pass
    # Identical copies are safe; different versions require an explicit path.
    identities={}
    for f in choices:
        if f.is_dir(): raw=(f/'DATASET_MANIFEST.json').read_bytes()
        else:
            with zipfile.ZipFile(f) as z: raw=z.read('uav_thesis/DATASET_MANIFEST.json')
        identities.setdefault(hashlib.sha256(raw).hexdigest(),[]).append(f)
    if len(identities)!=1: raise RuntimeError('Add Input đúng source Dataset; hoặc điền SOURCE_PATH: '+str(choices))
    SOURCE_PATH=next(iter(identities.values()))[0]
SOURCE_PATH=Path(SOURCE_PATH)
SOURCE_ROOT=Path('/kaggle/temp/bs5_v4_fresh_source/uav_thesis')
if SOURCE_PATH.is_dir(): expected=(SOURCE_PATH/'DATASET_MANIFEST.json').read_bytes()
else:
    with zipfile.ZipFile(SOURCE_PATH) as z: expected=z.read('uav_thesis/DATASET_MANIFEST.json')
if SOURCE_ROOT.exists():
    if (SOURCE_ROOT/'DATASET_MANIFEST.json').read_bytes()!=expected: raise RuntimeError('Working source khác Dataset. Giữ output và dùng phiên sạch.')
else:
    SOURCE_ROOT.parent.mkdir(parents=True,exist_ok=True)
    if SOURCE_PATH.is_dir(): shutil.copytree(SOURCE_PATH,SOURCE_ROOT)
    else:
        with zipfile.ZipFile(SOURCE_PATH) as z: extract_safe(z,SOURCE_ROOT.parent)
manifest=json.loads(expected)
for rel,digest in manifest.items():
    if sha(SOURCE_ROOT/rel)!=digest: raise RuntimeError('Source checksum failed: '+rel)
RUN_ENV=os.environ.copy()
RUN_ENV.update(PYTHONDONTWRITEBYTECODE='1',OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',CUBLAS_WORKSPACE_CONFIG=':4096:8')
RUN_ENV['PYTHONPATH']=str(SOURCE_ROOT)
probe="import torch,numpy,matplotlib,json; print(json.dumps(dict(torch=torch.__version__,numpy=numpy.__version__,matplotlib=matplotlib.__version__,cuda=torch.cuda.is_available())))"
check=subprocess.run([sys.executable,'-B','-c',probe],cwd=SOURCE_ROOT,env=RUN_ENV,capture_output=True,text=True)
if check.returncode: raise RuntimeError('Kaggle runtime cần torch/numpy/matplotlib. Không tự thay runtime khi resume. '+check.stderr[-2000:])
print('Verified source:',SOURCE_ROOT,'files:',len(manifest))
print('Installed runtime:',check.stdout.strip())
def run_stream(args, log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as f:
        proc=subprocess.Popen([sys.executable,'-B','-u',*map(str,args)],cwd=SOURCE_ROOT,env=RUN_ENV,
                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout: print(line,end='',flush=True);f.write(line);f.flush()
        return proc.wait()
def restore_archive_or_directory(src, destination, required):
    src,destination=Path(src),Path(destination)
    if destination.exists(): raise RuntimeError('Không chép đè output hiện có: '+str(destination))
    if src.is_dir() and (src/'protocol.json').exists():
        shutil.copytree(src,destination)
    else:
        if src.is_dir():
            candidates=list(src.rglob(required))
            if len(candidates)!=1: raise RuntimeError('Chọn đúng thư mục hoặc ZIP '+required+': '+str(candidates))
            src=candidates[0]
        with zipfile.ZipFile(src) as z:
            # Transport archives have exactly one top-level run directory.
            roots={Path(n).parts[0] for n in z.namelist() if Path(n).parts}
            if len(roots)!=1: raise RuntimeError('Resume ZIP phải có một thư mục gốc.')
            top=next(iter(roots))
            staging=WORK/('restore_'+destination.name)
            if staging.exists(): raise RuntimeError('Restore dở dang; giữ thư mục để kiểm tra: '+str(staging))
            staging.mkdir()
            extract_safe(z,staging)
            shutil.move(str(staging/top),str(destination));staging.rmdir()
    for checksum_name in ['EXPORT_SHA256SUMS.json','HANDOFF_SHA256SUMS.json']:
        checksum=destination/checksum_name
        if checksum.exists():
            for rel,digest in json.loads(checksum.read_text()).items():
                if sha(destination/rel)!=digest: raise RuntimeError('Input checksum failed: '+rel)
    if not (destination/'protocol.json').exists(): raise RuntimeError('Thiếu protocol trong resume.')
def show_exports(root):
    from IPython.display import display, FileLink, Markdown
    exports=sorted([p for p in WORK.glob(root.name+'_export*') if (p/'EXPORT_INDEX.json').exists()],key=lambda p:p.stat().st_mtime)
    if not exports: raise RuntimeError('Không có gói export hoàn tất; giữ nguyên output và log.')
    export=exports[-1]
    report=export/'analysis/REPORT.md'
    if report.exists(): display(Markdown(report.read_text()))
    for f in sorted(export.iterdir()):
        if f.is_file(): display(FileLink(str(f)))
    print('OUTPUT:',export)
    if json.loads((export/'EXPORT_INDEX.json').read_text()).get('export_audit')=='failed': raise RuntimeError('Export audit failed; raw evidence archive has been saved above.')
    return export

runtime_info=json.loads(check.stdout)
if REQUIRE_GPU and not runtime_info['cuda']:
    raise RuntimeError('GPU chưa sẵn sàng hoặc đang dùng PyTorch +cpu. Bật Accelerator GPU và dùng runtime Kaggle có Torch CUDA trước khi train; không đổi Torch/device giữa chừng khi resume.')

In [ ]:
RESUME_NAME='training_resume__'+JOB_LABEL+'.zip'
if RESUME_INPUT_DIR is None and not OUTPUT_ROOT.exists():
    archives=[x for x in INPUT.rglob(RESUME_NAME) if is_main_archive(x,'bs5_return_v4_fresh_crcn_v1')]
    directories=[]
    for f in INPUT.rglob('protocol.json'):
        try:
            p=json.loads(f.read_text())
            if p.get('version')=='bs5_return_v4_fresh_crcn_v1' and not p.get('diagnostic',False) and p.get('arms')==TRAIN_ARMS and p.get('training_seeds')==TRAINING_SEEDS and list((f.parent/'runs').glob('*/resume/*.pt')): directories.append(f.parent)
        except (OSError,ValueError): pass
    choices=archives if archives else directories
    if len(choices)>1: raise RuntimeError('Có nhiều resume cùng job; điền RESUME_INPUT_DIR rõ ràng.')
    if choices: RESUME_INPUT_DIR=choices[0]
if RESUME_INPUT_DIR is not None and not OUTPUT_ROOT.exists(): restore_archive_or_directory(RESUME_INPUT_DIR,OUTPUT_ROOT,RESUME_NAME)
if OUTPUT_ROOT.exists():
    saved=json.loads((OUTPUT_ROOT/'protocol.json').read_text())
    assert saved['arms']==TRAIN_ARMS and saved['training_seeds']==TRAINING_SEEDS,'Resume thuộc job khác.'
PREFLIGHT=WORK/('preflight_'+JOB_LABEL)
rc=run_stream(['scripts/verification/preflight_bs5_v4_partition.py','--output',PREFLIGHT,'--arms',','.join(TRAIN_ARMS),'--seeds',','.join(map(str,TRAINING_SEEDS)),'--device',DEVICE],WORK/('preflight_'+JOB_LABEL+'.log'))
if rc: raise RuntimeError('Preflight failed; main chưa chạy. Giữ log/receipt.')
receipt=json.loads((PREFLIGHT/'receipt.json').read_text())
assert receipt['status']=='passed'
print('Preflight PASS:',receipt['identity']['runtime']['device'],'— không chạy benchmark; chỉ dùng seed chẩn đoán riêng.')

In [ ]:
MAIN_RETURN_CODE=0
if RUN_MAIN:
    if not OUTPUT_ROOT.exists():
        rc=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','prepare','--output',OUTPUT_ROOT,'--device',DEVICE,'--arms',','.join(TRAIN_ARMS),'--seeds',','.join(map(str,TRAINING_SEEDS))],WORK/'bs5_v4_prepare.log')
        if rc: raise RuntimeError('Prepare failed; giữ log.')
    preflight=OUTPUT_ROOT/'preflight';preflight.mkdir(exist_ok=True)
    shutil.copy2(PREFLIGHT/'receipt.json',preflight/'receipt.json')
    shutil.copy2(WORK/('preflight_'+JOB_LABEL+'.log'),preflight/'preflight.log')
    budget=max(0,MAX_TOTAL_SESSION_SECONDS-EXPORT_RESERVE_SECONDS-(time.monotonic()-SESSION_START))
    print('Main soft budget remaining:',round(budget/3600,2),'hours')
    MAIN_RETURN_CODE=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','run','--output',OUTPUT_ROOT,'--max-seconds',str(budget)],WORK/'bs5_v4_training.log')
    print(json.loads((OUTPUT_ROOT/'status.json').read_text()))
else: print('RUN_MAIN=False: bounded diagnostic preflight only.')

In [ ]:
if RUN_MAIN and (OUTPUT_ROOT/'protocol.json').exists():
    rc=run_stream(['scripts/training/run_bs5_v4_fresh_train.py','export','--output',OUTPUT_ROOT],WORK/'bs5_v4_export.log')
    EXPORT=show_exports(OUTPUT_ROOT)
    if rc: raise RuntimeError('Export audit failed; raw output vẫn được giữ. Xem log trước khi tiếp tục.')
    if MAIN_RETURN_CODE: raise RuntimeError('Main có lỗi; gói bằng chứng đã xuất. Không xóa/sửa ledger để chạy lại.')

## Output riêng cho CN_552

- `review_small.zip`: report, learning curve, raw-bin/episode/update/validation tables.
- `training_resume__CN_552.zip`: resume ĐÚNG job này; giữ đủ optimizer/replay/RNG. Phiên sau Add Input source Dataset và gói resume này.
- `training_for_benchmark__CN_552.zip`: chỉ xuất khi các run ĐƯỢC GÁN đã hoàn tất. Có best/last, lựa chọn validation, curve CSV và hash UE tape. Không dùng để resume training.
- Lưu notebook version kèm output. Không trộn gói này với output source Dataset cũ.
- Runtime, device và loại GPU phải giữ nguyên trong cùng một job khi resume. Hai job độc lập có thể dùng hai GPU model khác nhau, được ghi đầy đủ; Python/Torch/NumPy stack phải khớp để ghép so sánh.
- Khi đủ bốn job, Add Input các gói `training_for_benchmark__*.zip` vào notebook `bs5_return_v4_parallel_benchmark_kaggle.ipynb`. Nó kiểm đủ job, recipe, initial weights và UE hashes rồi ghép learning curve trước benchmark.
- Không có test10k trong notebook train này. Không dùng test để chọn checkpoint. S1/S2 là fixture inherited dạy Relay; C-N là đối chứng cùng recipe.